In [22]:
import pandas as pd
from training.static_data import StaticData
from datetime import date

myDate = date(2024, 6, 15)
folder_path = "../../../../data"

koda_static_path = f"{folder_path}/koda-static/data-tmp"

# Load data similarly to api
try:
    static_data = StaticData.load_from_pkl(koda_static_path)
except FileNotFoundError:
    static_data = StaticData.load_static_data(koda_static_path, myDate)

    static_data.save_to_pkl(koda_static_path)

route_short_name = "4"
SL_AGENCY_ID = static_data.agencies[static_data.agencies["agency_name"] == "AB Storstockholms Lokaltrafik"].index[0]
route_subset = static_data.routes[
    (static_data.routes['route_short_name'] == route_short_name) & \
    (static_data.routes["agency_id"] == SL_AGENCY_ID)]

if not route_subset.empty:
    route_id = route_subset.index[0]
    print(f"Route ID: {route_id}")
    
    trips_for_route = static_data.trips[static_data.trips["route_id"] == route_id]
    print(f"Trips count: {len(trips_for_route)}")
    
    trip_ids = trips_for_route.index
    
    # Filter stop times
    relevant_stop_times = static_data.stop_times[
        static_data.stop_times.index.get_level_values(0).isin(trip_ids)
    ].sort_index()
    
    # Check sequences
    sequences = relevant_stop_times.groupby("trip_id")["stop_id"].apply(tuple)
    print("Top sequences:")
    print(sequences.value_counts().head())
    
    most_common = sequences.value_counts().index[0]
    print(f"Most common length: {len(most_common)}")

    # Resolve stops
    stops_indexed = static_data.stops.set_index("stop_id")
    for sid in most_common[:5]:
        print(stops_indexed.loc[sid])
else:
    print("Route not found")


Route ID: 9011001000400000
Trips count: 1809
Top sequences:
stop_id
(9022001011725004, 9022001010261002, 9022001010401001, 9022001010136002, 9022001010661002, 9022001010658001, 9022001010655002, 9022001010651002, 9022001010649002, 9022001010645002, 9022001010369001, 9022001010367002, 9022001010363004, 9022001010183002, 9022001010151003, 9022001010188002, 9022001010191002, 9022001010193001, 9022001010194004, 9022001010199002, 9022001010198002, 9022001010045002, 9022001010203001, 9022001010627002, 9022001010098003)                                                          455
(9022001010098003, 9022001010627001, 9022001010203002, 9022001010045001, 9022001010040001, 9022001010198001, 9022001010199001, 9022001010194006, 9022001010192001, 9022001010191001, 9022001010188001, 9022001010151002, 9022001010183003, 9022001010363005, 9022001010367001, 9022001010369008, 9022001010421001, 9022001010645001, 9022001010647001, 9022001010649001, 9022001010651001, 9022001010655001, 9022001010657001, 90220

In [23]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [24]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [25]:
from datetime import date
from training.common import FeedID, Operator

operator = Operator.SL
myDate = date(2024, 6, 15)
hour = 10
feedId = FeedID.TripUpdates

In [26]:
from training.koda import download_koda_rt_file, download_koda_static_file

KODA_API_KEY = os.getenv("KODA_API_KEY")
if not KODA_API_KEY:
    raise ValueError("KODA_API_KEY not found in environment variables")

download_koda_rt_file(operator, feedId, myDate, api_key=KODA_API_KEY, data_dir=f"{folder_path}/koda-rt")
download_koda_rt_file(operator, FeedID.VehiclePositions, myDate, api_key=KODA_API_KEY, data_dir=f"{folder_path}/koda-rt")

download_koda_static_file(operator, myDate, api_key=KODA_API_KEY, data_dir=f"{folder_path}/koda-static")

File ../../../../data/koda-rt/sl_2024-06-15_TripUpdates.7z already exists. Skipping download.
File ../../../../data/koda-static/sl_2024-06-15.zip already exists. Skipping download.


200

In [27]:
import pickle
from training.static_data import StaticData

static_data: StaticData
pickle_file_path = f"{folder_path}/koda-static/static-data-{myDate.year:04d}-{myDate.month:02d}-{myDate.day:02d}.pkl"
if os.path.exists(pickle_file_path):
    print("Static data pickle file exists.")

    static_data = pickle.load(open(pickle_file_path, "rb"))
else:
    print("Loading static data from GTFS files.")
    static_data = StaticData.load_static_data(f"{folder_path}/koda-static/data-tmp", myDate)
    pickle.dump(static_data, open(pickle_file_path, "wb"))
static_data

Static data pickle file exists.


In [28]:
print(static_data.stop_times.dtypes)
static_data.stop_times.head()

arrival_time           datetime64[ns]
departure_time         datetime64[ns]
stop_id                string[python]
stop_headsign          string[python]
pickup_type                  category
drop_off_type                category
shape_dist_traveled           float64
timepoint                       int64
dtype: object


arrival_time      departure_time  \
trip_id           stop_sequence                                           
14010000631461521 1             2024-06-15 05:30:00 2024-06-15 05:30:00   
                  2             2024-06-15 05:31:02 2024-06-15 05:31:02   
                  3             2024-06-15 05:32:14 2024-06-15 05:32:14   
                  4             2024-06-15 05:32:51 2024-06-15 05:32:51   
                  5             2024-06-15 05:33:44 2024-06-15 05:33:44   

                                          stop_id   stop_headsign pickup_type  \
trip_id           stop_sequence                                                 
14010000631461521 1              9022001010369006  Stora Essingen           3   
                  2              9022001010421001  Stora Essingen           3   
                  3              9022001010645003  Stora Essingen           3   
                  4              9022001010462002  Stora Essingen           3   
                  5              9022001010449001  Stora Essingen           3   

                                drop_off_type  shape_dist_traveled  timepoint  
trip_id           stop_sequence                                                
14010000631461521 1                         1                 0.00          1  
                  2                         3               430.80          0  
                  3                         3               828.25          0  
                  4                         3              1032.70          0  
                  5                         3              1327.70          0

In [29]:
from training.gtfs import  load_pb_file

koda_rt_example_path = f"{folder_path}/koda-rt/data-tmp/sl/{feedId.value}/{myDate.year:04d}/{myDate.month:02d}/{myDate.day:02d}/{hour:02d}"

files_in_data_folder = os.listdir(koda_rt_example_path)
print("Files in data folder:", files_in_data_folder)

first_file_path = os.path.join(koda_rt_example_path, files_in_data_folder[0])

gtfs_feed_message = load_pb_file(first_file_path)

Files in data folder: ['sl-tripupdates-2024-06-15T10-00-03Z.pb', 'sl-tripupdates-2024-06-15T10-00-17Z.pb', 'sl-tripupdates-2024-06-15T10-00-31Z.pb', 'sl-tripupdates-2024-06-15T10-00-45Z.pb', 'sl-tripupdates-2024-06-15T10-01-13Z.pb', 'sl-tripupdates-2024-06-15T10-01-27Z.pb', 'sl-tripupdates-2024-06-15T10-01-41Z.pb', 'sl-tripupdates-2024-06-15T10-01-55Z.pb', 'sl-tripupdates-2024-06-15T10-02-09Z.pb', 'sl-tripupdates-2024-06-15T10-02-23Z.pb', 'sl-tripupdates-2024-06-15T10-02-37Z.pb', 'sl-tripupdates-2024-06-15T10-02-51Z.pb', 'sl-tripupdates-2024-06-15T10-03-05Z.pb', 'sl-tripupdates-2024-06-15T10-03-19Z.pb', 'sl-tripupdates-2024-06-15T10-03-33Z.pb', 'sl-tripupdates-2024-06-15T10-03-47Z.pb', 'sl-tripupdates-2024-06-15T10-04-01Z.pb', 'sl-tripupdates-2024-06-15T10-04-15Z.pb', 'sl-tripupdates-2024-06-15T10-04-29Z.pb', 'sl-tripupdates-2024-06-15T10-04-43Z.pb', 'sl-tripupdates-2024-06-15T10-05-11Z.pb', 'sl-tripupdates-2024-06-15T10-05-25Z.pb', 'sl-tripupdates-2024-06-15T10-05-39Z.pb', 'sl-tripupd

In [30]:
from training.data_processing import feed_message_to_trip_update_dataframe

df = feed_message_to_trip_update_dataframe(gtfs_feed_message)

print(df.dtypes)

print("Number of rows in DataFrame:", len(df))

df.head()

id                                Int64
trip_id                  string[python]
start_date                       object
schedule_relationship             int64
vehicle_id                        Int64
stop_time_updates                object
timestamp                         int64
dtype: object
Number of rows in DataFrame: 689


,id,trip_id,start_date,schedule_relationship,vehicle_id,stop_time_updates,timestamp
0,14010516247726276,14010000663741929,20240615,0,9031008000500536,"[{'stop_sequence': 25, 'stop_id': '90220010001...",1718438398
1,14010516247729020,14010000663747674,20240615,0,9031008000500539,"[{'stop_sequence': 11, 'stop_id': '90220010001...",1718438398
2,14010516089523031,14010000656705623,20240615,0,9031001004002220,"[{'stop_sequence': 24, 'stop_id': '90220010050...",1718438398
3,14010516113220145,14010000656788822,20240615,0,9031001004302521,"[{'stop_sequence': 21, 'stop_id': '90220010061...",1718438398
4,14010516089526040,14010000656705875,20240615,0,9031001004002221,"[{'stop_sequence': 21, 'stop_id': '90220010051...",1718438398


In [31]:
from training.data_processing import join_static_data_on_rt_trip_updates


df = join_static_data_on_rt_trip_updates(static_data, df)

In [32]:
df[(df["route_short_name"] == "40") & (df["route_desc"] == "Pendeltåg")].head()

,id,trip_id,start_date,schedule_relationship,vehicle_id,stop_time_updates,timestamp,route_id,service_id,trip_headsign,direction_id,shape_id,agency_id,route_short_name,route_long_name,route_type,route_desc
2,14010516089523031,14010000656705623,20240615,0,9031001004002220,"[{'stop_sequence': 24, 'stop_id': '90220010050...",1718438398,9011001004000000,6,<NA>,0.0,4014010000492969507,14010000000001001,40,<NA>,100,Pendeltåg
4,14010516089526040,14010000656705875,20240615,0,9031001004002221,"[{'stop_sequence': 21, 'stop_id': '90220010051...",1718438398,9011001004000000,332,<NA>,1.0,4014010000492969547,14010000000001001,40,<NA>,100,Pendeltåg
19,14010516089527825,14010000656705974,20240615,0,9031001004002222,"[{'stop_sequence': 17, 'stop_id': '90220010050...",1718438398,9011001004000000,433,<NA>,0.0,4014010000492969316,14010000000001001,40,<NA>,100,Pendeltåg
25,14010516089529610,14010000656706199,20240615,0,9031001004002223,"[{'stop_sequence': 12, 'stop_id': '90220010053...",1718438398,9011001004000000,332,<NA>,1.0,4014010000492969547,14010000000001001,40,<NA>,100,Pendeltåg
113,14010516089531701,14010000656706391,20240615,0,9031001004002224,"[{'stop_sequence': 8, 'stop_id': '902200100516...",1718438398,9011001004000000,6,<NA>,0.0,4014010000492969507,14010000000001001,40,<NA>,100,Pendeltåg


In [33]:
from training.data_processing import explode_to_stops_with_join_static

df_exploded_with_stop_times = explode_to_stops_with_join_static(static_data, df)

print(df_exploded_with_stop_times.dtypes)

df_exploded_with_stop_times.head()

id                                           Int64
start_date                                  object
schedule_relationship                        int64
vehicle_id                                   Int64
timestamp                                    int64
route_id                                    object
service_id                                   Int64
trip_headsign                       string[python]
direction_id                               float64
shape_id                                     Int64
agency_id                           string[python]
route_short_name                    string[python]
route_long_name                     string[python]
route_type                                   Int64
route_desc                          string[python]
stop_id                             string[python]
arrival_time                         datetime64[s]
departure_time                       datetime64[s]
stop_time_schedule_relationship              int64
arrival_time_planned           

id start_date  \
trip_id           stop_sequence                                 
14010000663741929 25             14010516247726276   20240615   
                  26             14010516247726276   20240615   
                  27             14010516247726276   20240615   
                  28             14010516247726276   20240615   
                  29             14010516247726276   20240615   

                                 schedule_relationship        vehicle_id  \
trip_id           stop_sequence                                            
14010000663741929 25                                 0  9031008000500536   
                  26                                 0  9031008000500536   
                  27                                 0  9031008000500536   
                  28                                 0  9031008000500536   
                  29                                 0  9031008000500536   

                                  timestamp          route_id  service_id  \
trip_id           stop_sequence                                             
14010000663741929 25             1718438398  9011008001300000           6   
                  26             1718438398  9011008001300000           6   
                  27             1718438398  9011008001300000           6   
                  28             1718438398  9011008001300000           6   
                  29             1718438398  9011008001300000           6   

                                trip_headsign  direction_id  \
trip_id           stop_sequence                               
14010000663741929 25                     <NA>           0.0   
                  26                     <NA>           0.0   
                  27                     <NA>           0.0   
                  28                     <NA>           0.0   
                  29                     <NA>           0.0   

                                            shape_id          agency_id  \
trip_id           stop_sequence                                           
14010000663741929 25             6014010000657796244  14010000000002071   
                  26             6014010000657796244  14010000000002071   
                  27             6014010000657796244  14010000000002071   
                  28             6014010000657796244  14010000000002071   
                  29             6014010000657796244  14010000000002071   

                                route_short_name route_long_name  route_type  \
trip_id           stop_sequence                                                
14010000663741929 25                          13            <NA>        1000   
                  26                          13            <NA>        1000   
                  27                          13            <NA>        1000   
                  28                          13            <NA>        1000   
                  29                          13            <NA>        1000   

                                      route_desc           stop_id  \
trip_id           stop_sequence                                      
14010000663741929 25             Waxholmsbolaget  9022001000143001   
                  26             Waxholmsbolaget  9022001000142001   
                  27             Waxholmsbolaget  9022001000141001   
                  28             Waxholmsbolaget  9022001000139001   
                  29             Waxholmsbolaget  9022001000138001   

                                       arrival_time      departure_time  \
trip_id           stop_sequence                                           
14010000663741929 25            2024-06-15 07:51:16 2024-06-15 07:52:13   
                  26            2024-06-15 07:53:16 2024-06-15 07:53:16   
                  27            2024-06-15 07:55:06 2024-06-15 07:55:49   
                  28            2024-06-15 07:59:29 2024-06-15 07:59:29   
                  29            2024-06-15 07:59:

In [34]:
# df_exploded_with_stop_times["arrival_time_late"] = (df_exploded_with_stop_times["arrival_time"] - df_exploded_with_stop_times["arrival_time_planned"])
# df_exploded_with_stop_times["departure_time_late"] = (df_exploded_with_stop_times["departure_time"] - df_exploded_with_stop_times["departure_time_planned"])

print(df_exploded_with_stop_times[["arrival_time", "arrival_time_planned", "arrival_time_late",
                             "departure_time", "departure_time_planned", "departure_time_late", "agency_id"]].dtypes)

df_exploded_with_stop_times[["arrival_time", "arrival_time_planned", "arrival_time_late",
                             "departure_time", "departure_time_planned", "departure_time_late", "agency_id"]].head()

arrival_time                datetime64[s]
arrival_time_planned       datetime64[ns]
arrival_time_late         timedelta64[ns]
departure_time              datetime64[s]
departure_time_planned     datetime64[ns]
departure_time_late       timedelta64[ns]
agency_id                  string[python]
dtype: object


arrival_time arrival_time_planned  \
trip_id           stop_sequence                                            
14010000663741929 25            2024-06-15 07:51:16  2024-06-15 07:46:00   
                  26            2024-06-15 07:53:16  2024-06-15 07:47:00   
                  27            2024-06-15 07:55:06  2024-06-15 07:50:00   
                  28            2024-06-15 07:59:29  2024-06-15 07:53:00   
                  29            2024-06-15 07:59:59  2024-06-15 07:54:00   

                                arrival_time_late      departure_time  \
trip_id           stop_sequence                                         
14010000663741929 25              0 days 00:05:16 2024-06-15 07:52:13   
                  26              0 days 00:06:16 2024-06-15 07:53:16   
                  27              0 days 00:05:06 2024-06-15 07:55:49   
                  28              0 days 00:06:29 2024-06-15 07:59:29   
                  29              0 days 00:05:59 2024-06-15 08:00:01   

                                departure_time_planned departure_time_late  \
trip_id           stop_sequence                                              
14010000663741929 25               2024-06-15 07:46:00     0 days 00:06:13   
                  26               2024-06-15 07:47:00     0 days 00:06:16   
                  27               2024-06-15 07:50:00     0 days 00:05:49   
                  28               2024-06-15 07:53:00     0 days 00:06:29   
                  29               2024-06-15 07:54:00     0 days 00:06:01   

                                         agency_id  
trip_id           stop_sequence                     
14010000663741929 25             14010000000002071  
                  26             14010000000002071  
                  27             14010000000002071  
                  28             14010000000002071  
                  29             14010000000002071

In [35]:
SL_AGENCY_ID = static_data.agencies[static_data.agencies["agency_name"] == "AB Storstockholms Lokaltrafik"].index[0]
# WAXHOLMSBOLAGET_ID = static_data.agencies[static_data.agencies["agency_name"] == "Waxholmsbolaget Ångfartygs AB"].index[0]

# route_dt = df_exploded_with_stop_times[
#   (df_exploded_with_stop_times["route_short_name"] == "13") & \
#   (df_exploded_with_stop_times["agency_id"] == WAXHOLMSBOLAGET_ID)]
# route_dt.head(20)

In [36]:
df_exploded_with_stop_times[(df_exploded_with_stop_times["route_desc"] == "blåbuss") & \
  (df_exploded_with_stop_times["route_short_name"].isin(["1", "2", "3", "4"])) & \
  (df_exploded_with_stop_times["agency_id"] == SL_AGENCY_ID)]["route_short_name"].value_counts()

route_short_name
4    170
2    107
1     69
3     53
Name: count, dtype: Int64

In [37]:
df_exploded_with_stop_times[(df_exploded_with_stop_times["route_desc"] == "blåbuss") & \
  (df_exploded_with_stop_times["route_short_name"].isin(["4"])) & \
  (df_exploded_with_stop_times["agency_id"] == SL_AGENCY_ID)].head(20)

id start_date  \
trip_id           stop_sequence                                 
14010000637145800 21             14010515937867392   20240615   
                  22             14010515937867392   20240615   
                  23             14010515937867392   20240615   
                  24             14010515937867392   20240615   
                  25             14010515937867392   20240615   
14010000641290683 15             14010515950004186   20240615   
                  16             14010515950004186   20240615   
                  17             14010515950004186   20240615   
                  18             14010515950004186   20240615   
                  19             14010515950004186   20240615   
                  20             14010515950004186   20240615   
                  21             14010515950004186   20240615   
                  22             14010515950004186   20240615   
                  23             14010515950004186   20240615   
                  24             14010515950004186   20240615   
                  25             14010515950004186   20240615   
14010000641290864 16             14010515949994700   20240615   
                  17             14010515949994700   20240615   
                  18             14010515949994700   20240615   
                  19             14010515949994700   20240615   

                                 schedule_relationship        vehicle_id  \
trip_id           stop_sequence                                            
14010000637145800 21                                 0  9031001001004010   
                  22                                 0  9031001001004010   
                  23                                 0  9031001001004010   
                  24                                 0  9031001001004010   
                  25                                 0  9031001001004010   
14010000641290683 15                                 0  9031001001001555   
                  16                                 0  9031001001001555   
                  17                                 0  9031001001001555   
                  18                                 0  9031001001001555   
                  19                                 0  9031001001001555   
                  20                                 0  9031001001001555   
                  21                                 0  9031001001001555   
                  22                                 0  9031001001001555   
                  23                                 0  9031001001001555   
                  24                                 0  9031001001001555   
                  25                                 0  9031001001001555   
14010000641290864 16                                 0  9031001001001553   
                  17                                 0  9031001001001553   
                  18                                 0  9031001001001553   
                  19                                 0  9031001001001553   

                                  timestamp          route_id  service_id  \
trip_id           stop_sequence                                             
14010000637145800 21             1718438398  9011001000400000           6   
                  22             1718438398  9011001000400000           6   
                  23             1718438398  9011001000400000           6   
                  24             1718438398  9011001000400000           6   
                  25             1718438398  9011001000400000           6   
14010000641290683 15             1718438398  9011001000400000           6   
                  16             1718438398  9011001000400000           6   
                  17             1718438398  9011001000400000           6   
                  18             1718438398  9011001000400000           6   
                  19             1718438398  9011001000400000           6   
                  20

In [38]:
df_exploded_with_stop_times = df_exploded_with_stop_times.sort_values(['trip_id', 'stop_sequence'])

df_exploded_with_stop_times['arrival_time_prev'] = df_exploded_with_stop_times.groupby('trip_id')['arrival_time'].shift(1)
df_exploded_with_stop_times['departure_time_prev'] = df_exploded_with_stop_times.groupby('trip_id')['departure_time'].shift(1)
df_exploded_with_stop_times['arrival_time_planned_prev'] = df_exploded_with_stop_times.groupby('trip_id')['arrival_time_planned'].shift(1)
df_exploded_with_stop_times['departure_time_planned_prev'] = df_exploded_with_stop_times.groupby('trip_id')['departure_time_planned'].shift(1)
df_exploded_with_stop_times['arrival_time_late_prev'] = df_exploded_with_stop_times.groupby('trip_id')['arrival_time_late'].shift(1)
df_exploded_with_stop_times['departure_time_late_prev'] = df_exploded_with_stop_times.groupby('trip_id')['departure_time_late'].shift(1)

df_exploded_with_stop_times['arrival_time_next'] = df_exploded_with_stop_times.groupby('trip_id')['arrival_time'].shift(-1)
df_exploded_with_stop_times['departure_time_next'] = df_exploded_with_stop_times.groupby('trip_id')['departure_time'].shift(-1)
df_exploded_with_stop_times['arrival_time_planned_next'] = df_exploded_with_stop_times.groupby('trip_id')['arrival_time_planned'].shift(-1)
df_exploded_with_stop_times['departure_time_planned_next'] = df_exploded_with_stop_times.groupby('trip_id')['departure_time_planned'].shift(-1)
df_exploded_with_stop_times['arrival_time_late_next'] = df_exploded_with_stop_times.groupby('trip_id')['arrival_time_late'].shift(-1)
df_exploded_with_stop_times['departure_time_late_next'] = df_exploded_with_stop_times.groupby('trip_id')['departure_time_late'].shift(-1)
df_exploded_with_stop_times.head(40)

id start_date  \
trip_id           stop_sequence                                 
14010000496968823 1              14010515507510490   20240615   
                  2              14010515507510490   20240615   
                  3              14010515507510490   20240615   
                  4              14010515507510490   20240615   
                  5              14010515507510490   20240615   
                  6              14010515507510490   20240615   
                  7              14010515507510490   20240615   
                  8              14010515507510490   20240615   
                  9              14010515507510490   20240615   
                  10             14010515507510490   20240615   
                  11             14010515507510490   20240615   
                  12             14010515507510490   20240615   
                  13             14010515507510490   20240615   
                  14             14010515507510490   20240615   
14010000496968868 1              14010515507510606   20240615   
                  2              14010515507510606   20240615   
                  3              14010515507510606   20240615   
                  4              14010515507510606   20240615   
                  5              14010515507510606   20240615   
                  6              14010515507510606   20240615   
                  7              14010515507510606   20240615   
                  8              14010515507510606   20240615   
                  9              14010515507510606   20240615   
                  10             14010515507510606   20240615   
                  11             14010515507510606   20240615   
                  12             14010515507510606   20240615   
                  13             14010515507510606   20240615   
                  14             14010515507510606   20240615   
14010000502525650 6              14010515507510258   20240615   
                  7              14010515507510258   20240615   
                  8              14010515507510258   20240615   
                  9              14010515507510258   20240615   
                  10             14010515507510258   20240615   
                  11             14010515507510258   20240615   
                  12             14010515507510258   20240615   
                  13             14010515507510258   20240615   
                  14             14010515507510258   20240615   
14010000502525695 1              14010515507510374   20240615   
                  2              14010515507510374   20240615   
                  3              14010515507510374   20240615   

                                 schedule_relationship        vehicle_id  \
trip_id           stop_sequence                                            
14010000496968823 1                                  0  9031001002510015   
                  2                                  0  9031001002510015   
                  3                                  0  9031001002510015   
                  4                                  0  9031001002510015   
                  5                                  0  9031001002510015   
                  6                                  0  9031001002510015   
                  7                                  0  9031001002510015   
                  8                                  0  9031001002510015   
                  9                                  0  9031001002510015   
                  10                                 0  9031001002510015   
                  11                                 0  9031001002510015   
                  12                                 0  9031001002510015   
                  13                                 0  9031001002510015   
                  14                                 0  9031001002510015   
14010000496968868 1                                  0  9031001002510003   
                  2      

In [39]:
from training.gtfs import download_gtfs_static_file

GTFS_REGIONAL_STATIC_API_KEY = os.getenv("GTFS_REGIONAL_STATIC_API_KEY")
if not GTFS_REGIONAL_STATIC_API_KEY:
    raise ValueError("GTFS_REGIONAL_STATIC_API_KEY not found in environment variables")

download_gtfs_static_file(Operator.SL, api_key=GTFS_REGIONAL_STATIC_API_KEY, data_dir=f"{folder_path}/gtfs-static")

File ../../../../data/gtfs-static/sl_gtfs_static.zip already exists. Skipping download.


In [40]:
from training.gtfs import download_gtfs_rt_file

GTFS_REGIONAL_RT_API_KEY = os.getenv("GTFS_REGIONAL_RT_API_KEY")
if not GTFS_REGIONAL_RT_API_KEY:
    raise ValueError("GTFS_REGIONAL_RT_API_KEY not found in environment variables")

download_gtfs_rt_file(Operator.SL, FeedID.TripUpdates, api_key=GTFS_REGIONAL_RT_API_KEY, data_dir=f"{folder_path}/gtfs-rt")
download_gtfs_rt_file(Operator.SL, FeedID.VehiclePositions, api_key=GTFS_REGIONAL_RT_API_KEY, data_dir=f"{folder_path}/gtfs-rt")

File ../../../../data/gtfs-rt/sl_TripUpdates.pb already exists. Skipping download.
File ../../../../data/gtfs-rt/sl_VehiclePositions.pb already exists. Skipping download.


In [41]:
todayDate = date.today()

gtfs_static_path = f"{folder_path}/gtfs-static/data-tmp"

# Load data similarly to api
try:
    gtfs_static_data = StaticData.load_from_pkl(gtfs_static_path)
except FileNotFoundError:
    gtfs_static_data = StaticData.load_static_data(gtfs_static_path, todayDate)

    gtfs_static_data.save_to_pkl(gtfs_static_path)

In [42]:
first_file_path = os.path.join(folder_path, "gtfs-rt/sl_VehiclePositions.pb")

gtfs_feed_message = load_pb_file(first_file_path)

In [43]:
from training.data_processing import feed_message_to_vehicle_position_dataframe, join_static_data_on_rt_vehicle_positions

df = feed_message_to_vehicle_position_dataframe(gtfs_feed_message)
df = join_static_data_on_rt_vehicle_positions(gtfs_static_data, df)

print(df.dtypes)

print("Number of rows in DataFrame:", len(df))

df.head()

id                                       Int64
trip_id                         string[python]
timestamp                                int64
vehicle_latitude                       float64
vehicle_longitude                      float64
vehicle_bearing                        float64
vehicle_odometer                       float64
vehicle_speed                          float64
vehicle_congestion_level                 int64
vehicle_occupancy_percentage             int64
vehicle_occupancy_status                 int64
stop_id                                 object
current_status                           int64
current_stop_sequence                    int64
route_id                                object
service_id                               Int64
trip_headsign                   string[python]
direction_id                           float64
shape_id                                 Int64
agency_id                       string[python]
route_short_name                string[python]
route_long_na

,id,trip_id,timestamp,vehicle_latitude,vehicle_longitude,vehicle_bearing,vehicle_odometer,vehicle_speed,vehicle_congestion_level,vehicle_occupancy_percentage,vehicle_occupancy_status,stop_id,current_status,current_stop_sequence,route_id,service_id,trip_headsign,direction_id,shape_id,agency_id,route_short_name,route_long_name,route_type,route_desc
0,51161768139111764,14010000710657261,1768139423,60.207085,18.738403,314.0,0.0,0.0,0,0,0,,2,0,9011001063700000,46,<NA>,1.0,1014010000406193844,14010000000001001,637,<NA>,700,<NA>
1,22431768139379372,14010000668240205,1768139423,59.343212,18.045721,0.0,0.0,-0.3,0,0,0,,2,0,9011001004000000,85,<NA>,1.0,4014010000668234934,14010000000001001,40,<NA>,100,Pendeltåg
2,15401768139422865,14010000664258364,1768139423,59.314598,18.097952,299.0,0.0,0.0,0,0,0,,2,0,9011001005300000,210,<NA>,0.0,1014010000631382407,14010000000001001,53,<NA>,700,<NA>
3,53631768139422409,14010000684991503,1768139423,59.319073,18.283159,230.0,0.0,24.4,0,0,0,,2,0,9011001043800000,359,<NA>,0.0,1014010000510129890,14010000000001001,438,<NA>,700,<NA>
4,71831768139423053,14010000708551486,1768139423,59.372593,18.189606,111.0,0.0,4.7,0,0,0,,2,0,9011001021200000,267,<NA>,1.0,1014010000604411767,14010000000001001,212,<NA>,700,<NA>


In [47]:
SL_AGENCY_ID = gtfs_static_data.agencies[gtfs_static_data.agencies["agency_name"] == "AB Storstockholms Lokaltrafik"].index[0]

STAM_BUSES = ["1", "2", "3", "4"]

df[df["route_short_name"].isin(STAM_BUSES) & (df["agency_id"] == SL_AGENCY_ID)]["route_short_name"].value_counts()

route_short_name
4    11
1    11
3    10
2     9
Name: count, dtype: Int64